# 🦟 Malaria Incidence Prediction in Africa
## Mission: Combating Infectious Diseases in Rwanda, Uganda & Kenya

> **Mission Statement:** To leverage machine learning to predict malaria incidence rates across African nations — with a focus on Rwanda, Uganda, and Kenya — enabling governments and health authorities to proactively allocate resources, target high-risk regions, and reduce preventable malaria deaths.

**Dataset:** [Malaria in Africa – Kaggle (lydia70)](https://www.kaggle.com/datasets/lydia70/malaria-in-africa)  
**Source:** World Bank Open Data, cleaned and published on Kaggle.  
**Coverage:** All 54 African countries, 2007–2017.  
**Features:** Country, Year, malaria incidence per 1,000 population at risk, reported cases, insecticide-treated bed net usage (%), children with fever receiving antimalarial drugs (%), pregnant women receiving preventive treatment (%), and more — several columns contain real-world **null values** handled via imputation.

**Target Variable:** `Incidence of malaria (per 1,000 population at risk)`

In [ ]:
# ── 0. Install all dependencies ────────────────────────────────────────────
import subprocess, sys

packages = [
    'kagglehub[pandas-datasets]',
    'pandas', 'numpy', 'matplotlib',
    'seaborn', 'scikit-learn', 'joblib', 'scipy'
]
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, joblib, os
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('✅ All libraries loaded successfully')

---
## 1. Load Dataset from Kaggle via `kagglehub`

No Kaggle credentials needed — `kagglehub` handles authentication automatically via your environment.

In [ ]:
# ── 1. Load dataset using kagglehub ───────────────────────────────────────
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = ""

# Load the latest version
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "lydia70/malaria-in-africa",
    file_path,
)

print("First 5 records:", df.head())
print(f"\n✅ Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")

---
## 2. Exploratory Data Analysis (EDA)

In [ ]:
# ── 2a. Overview: shape, dtypes, missing values ────────────────────────────
df_raw = df.copy()

print('=== DATASET OVERVIEW ===')
print(f'Shape: {df_raw.shape}')
print(f'\nColumns & Data Types:')
for i, (col, dtype) in enumerate(df_raw.dtypes.items(), 1):
    print(f'  {i:2}. {col:<60} {str(dtype)}')

print(f'\n=== MISSING VALUES ===')
missing     = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df  = pd.DataFrame({'Count': missing, 'Percentage (%)': missing_pct})
print(missing_df[missing_df['Count'] > 0].to_string())
print(f'\nTotal missing cells: {missing.sum()}')

In [ ]:
# ── 2b. Statistical Summary ────────────────────────────────────────────────
df_raw.describe().round(2)

In [ ]:
# ── 2c. Detect key columns dynamically ────────────────────────────────────
incidence_col = [c for c in df_raw.columns if 'incidence' in c.lower()][0]
country_col   = [c for c in df_raw.columns
                 if 'country' in c.lower() and 'code' not in c.lower()][0]
year_col      = [c for c in df_raw.columns if 'year' in c.lower()][0]

print(f'Country column : "{country_col}"')
print(f'Year column    : "{year_col}"')
print(f'Target column  : "{incidence_col}"')
print(f'\nUnique countries : {df_raw[country_col].nunique()}')
print(f'Year range       : {int(df_raw[year_col].min())} – {int(df_raw[year_col].max())}')

In [ ]:
# ── 2d. Visualization 1 – Missing Values Heatmap ──────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(df_raw.isnull(), yticklabels=False, cbar=True, cmap='viridis', ax=ax)
ax.set_title('Visualization 1: Missing Values Heatmap – Malaria in Africa Dataset',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Columns')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('viz1_missing_values.png', dpi=150)
plt.show()

print()
print('📊 Interpretation:')
print('   Yellow stripes = missing values. Prevention columns (bed nets, fever')
print('   treatment %, antenatal care %) have the most gaps, reflecting inconsistent')
print('   health data collection across African countries. These will be handled')
print('   via median imputation in the preprocessing step.')

In [ ]:
# ── 2e. Visualization 2 – Malaria Incidence Over Time (East Africa) ────────
east_africa = ['Rwanda', 'Uganda', 'Kenya']
colors      = ['#E63946', '#2A9D8F', '#F4A261']

fig, ax = plt.subplots(figsize=(12, 5))
for country, color in zip(east_africa, colors):
    subset = df_raw[df_raw[country_col] == country].sort_values(year_col)
    if len(subset) > 0:
        ax.plot(subset[year_col], subset[incidence_col],
                marker='o', linewidth=2.2, label=country, color=color)

ax.set_title('Visualization 2: Malaria Incidence Over Time – Rwanda, Uganda, Kenya',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Incidence per 1,000 population at risk')
ax.legend()
plt.tight_layout()
plt.savefig('viz2_incidence_over_time.png', dpi=150)
plt.show()

print()
print('📊 Interpretation:')
print('   Uganda has the highest and most volatile malaria burden throughout the period.')
print('   Rwanda shows a strong and consistent decline, reflecting its national malaria')
print('   control program. Kenya is lower but improvement has plateaued, highlighting')
print('   the need for renewed targeted intervention.')

In [ ]:
# ── 2f. Visualization 3 – Correlation Heatmap ─────────────────────────────
numeric_df = df_raw.select_dtypes(include=[np.number])

fig, ax = plt.subplots(figsize=(11, 8))
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8}, annot_kws={'size': 8})
ax.set_title('Visualization 3: Feature Correlation Heatmap',
             fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('viz3_correlation_heatmap.png', dpi=150)
plt.show()

print()
print('📊 Interpretation:')
print('   Darker red = strong positive correlation with incidence (key predictors).')
print('   Darker blue = protective negative correlation (e.g. prevention measures).')
print('   This matrix drives feature selection — highly correlated features are')
print('   retained; near-zero columns signal low predictive value.')

In [ ]:
# ── 2g. Visualization 4 – Target Distribution Before & After Log Transform ─
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
valid = df_raw[incidence_col].dropna()

axes[0].hist(valid, bins=25, color='#E63946', edgecolor='white', alpha=0.85)
axes[0].set_title('Viz 4a: Incidence Distribution (Raw)', fontweight='bold')
axes[0].set_xlabel(incidence_col)
axes[0].set_ylabel('Frequency')

axes[1].hist(np.log1p(valid), bins=25, color='#2A9D8F', edgecolor='white', alpha=0.85)
axes[1].set_title('Viz 4b: Incidence Distribution (Log-transformed)', fontweight='bold')
axes[1].set_xlabel('log(1 + Incidence)')
axes[1].set_ylabel('Frequency')

plt.suptitle('Visualization 4: Target Variable Distribution – Raw vs Log-transformed',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('viz4_target_distribution.png', dpi=150)
plt.show()

print()
print('📊 Interpretation:')
print('   The raw incidence is strongly right-skewed — a handful of high-burden')
print('   countries dominate. Log-transformation normalises the distribution,')
print('   reducing outlier influence and stabilising gradient descent training.')

---
## 3. Feature Engineering & Preprocessing

In [ ]:
# ── 3a. Drop irrelevant / high-cardinality identifier columns ─────────────
df = df_raw.copy()

drop_cols = []
for col in df.select_dtypes(include='object').columns:
    if col in [country_col, year_col]:
        continue   # encode these, don't drop
    unique_ratio = df[col].nunique() / len(df)
    if unique_ratio > 0.5:  # near-unique text = identifier, not a feature
        drop_cols.append(col)

print(f'Dropping high-cardinality / identifier columns: {drop_cols}')
df.drop(columns=drop_cols, inplace=True, errors='ignore')
print(f'Shape after drop: {df.shape}')

In [ ]:
# ── 3b. Convert categorical columns to numeric (Label Encoding) ───────────
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col].astype(str))
    print(f'✅ Label-encoded: "{col}"')

print(f'\nAll dtypes are now numeric: {list(df.dtypes.unique())}')

In [ ]:
# ── 3c. Handle Null Values ─────────────────────────────────────────────────
TARGET = [c for c in df.columns if 'incidence' in c.lower()][0]
print(f'Target column: "{TARGET}"')
print(f'\nMissing values BEFORE imputation:')
print(df.isnull().sum()[df.isnull().sum() > 0])

# Step 1 — drop rows where the TARGET itself is null (can't train without a label)
before = len(df)
df.dropna(subset=[TARGET], inplace=True)
print(f'\nDropped {before - len(df)} rows where target was null.')

# Step 2 — impute remaining feature nulls with MEDIAN (robust to outliers)
from sklearn.impute import SimpleImputer
imputer   = SimpleImputer(strategy='median')
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

print(f'Missing values AFTER imputation : {df_imputed.isnull().sum().sum()}')
print(f'Final dataset shape             : {df_imputed.shape}')

In [ ]:
# ── 3d. Log-transform skewed features and the target ──────────────────────
from scipy.stats import skew

log_cols = []
for col in df_imputed.columns:
    if col == TARGET:
        continue
    if abs(skew(df_imputed[col])) > 1.0:
        df_imputed[col] = np.log1p(df_imputed[col].clip(lower=0))
        log_cols.append(col)

df_imputed[TARGET] = np.log1p(df_imputed[TARGET].clip(lower=0))

print(f'Log-transformed features : {log_cols}')
print(f'Log-transformed target   : "{TARGET}"')

In [ ]:
# ── 3e. Define features & target, split into train/test ───────────────────
features = [c for c in df_imputed.columns if c != TARGET]
print(f'Features ({len(features)}): {features}')
print(f'Target                   : "{TARGET}"')

X = df_imputed[features].values
y = df_imputed[TARGET].values

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42)

print(f'\nTrain set : {X_train.shape}')
print(f'Test  set : {X_test.shape}')

In [ ]:
# ── 3f. Standardize features (StandardScaler) ─────────────────────────────
from sklearn.preprocessing import StandardScaler

scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

# Save scaler and feature list for Task 2 API
joblib.dump(scaler,   'scaler.pkl')
joblib.dump(features, 'feature_names.pkl')

print('✅ Standardization complete')
print(f'   Mean ≈ 0 : {X_train_sc.mean():.6f}')
print(f'   Std  ≈ 1 : {X_train_sc.std():.6f}')
print('✅ Saved: scaler.pkl  |  feature_names.pkl')

---
## 4. Model Training

### 4a. Linear Regression – Manual Gradient Descent + scikit-learn

In [ ]:
# ── 4a-i. Manual Batch Gradient Descent ───────────────────────────────────
def gradient_descent(X, y, lr=0.05, epochs=500):
    """
    Batch gradient descent for linear regression.
    Returns: (theta, mse_history)
    """
    m, n  = X.shape
    theta = np.zeros(n + 1)
    X_b   = np.c_[np.ones((m, 1)), X]   # add bias column
    history = []
    for _ in range(epochs):
        y_hat    = X_b @ theta
        error    = y_hat - y
        gradient = (2 / m) * (X_b.T @ error)
        theta   -= lr * gradient
        history.append(np.mean(error ** 2))
    return theta, history

EPOCHS = 500
LR     = 0.05

theta_final, train_loss = gradient_descent(X_train_sc, y_train, lr=LR, epochs=EPOCHS)
_,            test_loss  = gradient_descent(X_test_sc,  y_test,  lr=LR, epochs=EPOCHS)

print(f'Gradient Descent — {EPOCHS} epochs  |  lr = {LR}')
print(f'  Final Train MSE : {train_loss[-1]:.6f}')
print(f'  Final Test  MSE : {test_loss[-1]:.6f}')

In [ ]:
# ── 4a-ii. Visualization 5 – Loss Curve (Train vs Test) ───────────────────
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(range(EPOCHS), train_loss, label='Train Loss (MSE)',
        color='#E63946', linewidth=2)
ax.plot(range(EPOCHS), test_loss,  label='Test Loss (MSE)',
        color='#2A9D8F', linewidth=2, linestyle='--')
ax.set_title('Visualization 5: Gradient Descent – Loss Curve (Train vs Test)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Mean Squared Error')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.savefig('viz5_loss_curve.png', dpi=150)
plt.show()

print()
print('📊 Interpretation:')
print('   Both curves converge smoothly with no divergence, confirming a suitable')
print('   learning rate. The narrow gap between train and test loss shows good')
print('   generalisation — the model is not overfitting to the training data.')

In [ ]:
# ── 4a-iii. scikit-learn Linear Regression ────────────────────────────────
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

lr_model  = LinearRegression()
lr_model.fit(X_train_sc, y_train)
y_pred_lr = lr_model.predict(X_test_sc)

mse_lr = mean_squared_error(y_test, y_pred_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr  = r2_score(y_test, y_pred_lr)
print(f'✅ Linear Regression  →  MSE: {mse_lr:.4f}  |  MAE: {mae_lr:.4f}  |  R²: {r2_lr:.4f}')

### 4b. Decision Tree Regressor

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt_model  = DecisionTreeRegressor(max_depth=6, min_samples_split=5, random_state=42)
dt_model.fit(X_train_sc, y_train)
y_pred_dt = dt_model.predict(X_test_sc)

mse_dt = mean_squared_error(y_test, y_pred_dt)
mae_dt = mean_absolute_error(y_test, y_pred_dt)
r2_dt  = r2_score(y_test, y_pred_dt)
print(f'✅ Decision Tree      →  MSE: {mse_dt:.4f}  |  MAE: {mae_dt:.4f}  |  R²: {r2_dt:.4f}')

### 4c. Random Forest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model  = RandomForestRegressor(
    n_estimators=200, max_depth=10,
    min_samples_split=4, random_state=42, n_jobs=-1)
rf_model.fit(X_train_sc, y_train)
y_pred_rf = rf_model.predict(X_test_sc)

mse_rf = mean_squared_error(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf  = r2_score(y_test, y_pred_rf)
print(f'✅ Random Forest      →  MSE: {mse_rf:.4f}  |  MAE: {mae_rf:.4f}  |  R²: {r2_rf:.4f}')

---
## 5. Model Comparison & Save Best Model

In [ ]:
# ── 5. Compare all models ──────────────────────────────────────────────────
results = {
    'Linear Regression': {'model': lr_model, 'mse': mse_lr, 'mae': mae_lr, 'r2': r2_lr, 'preds': y_pred_lr},
    'Decision Tree':     {'model': dt_model, 'mse': mse_dt, 'mae': mae_dt, 'r2': r2_dt, 'preds': y_pred_dt},
    'Random Forest':     {'model': rf_model, 'mse': mse_rf, 'mae': mae_rf, 'r2': r2_rf, 'preds': y_pred_rf},
}

best_mse   = min(v['mse'] for v in results.values())
best_name  = min(results, key=lambda k: results[k]['mse'])
best_model = results[best_name]['model']
best_preds = results[best_name]['preds']

print('=' * 72)
print(f'{"Model":<22} {"MSE":>12} {"MAE":>12} {"R²":>10}')
print('-' * 72)
for name, info in results.items():
    tag = '  ← 🏆 BEST' if info['mse'] == best_mse else ''
    print(f'{name:<22} {info["mse"]:>12.4f} {info["mae"]:>12.4f} {info["r2"]:>10.4f}{tag}')
print('=' * 72)

joblib.dump(best_model, 'best_model.pkl')
print(f'\n✅ Best model ({best_name}) saved → best_model.pkl')

---
## 6. Scatter Plot – Before & After Fitting the Regression Line

In [ ]:
# ── 6. Scatter – Before & After ───────────────────────────────────────────
# Auto-select the feature most correlated with the target on the test set
corr_vals = np.abs(
    pd.DataFrame(X_test_sc, columns=features)
      .corrwith(pd.Series(y_test)).values)
top_idx  = corr_vals.argmax()
top_feat = features[top_idx]
print(f'Top feature selected for plot: "{top_feat}"')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── BEFORE: raw scatter ────────────────────────────────────────────────────
axes[0].scatter(X_test_sc[:, top_idx], y_test,
                color='#264653', alpha=0.65, edgecolors='white',
                linewidth=0.4, s=55)
axes[0].set_title(f'BEFORE Fitting\n"{top_feat}" vs Malaria Incidence',
                  fontweight='bold')
axes[0].set_xlabel(f'{top_feat} (standardised)')
axes[0].set_ylabel('log(1 + Incidence per 1,000)')

# ── AFTER: scatter + fitted linear regression line ─────────────────────────
x_range = np.linspace(X_test_sc[:, top_idx].min(),
                      X_test_sc[:, top_idx].max(), 300)
X_line  = np.tile(X_test_sc.mean(axis=0), (300, 1))
X_line[:, top_idx] = x_range
y_line  = lr_model.predict(X_line)

axes[1].scatter(X_test_sc[:, top_idx], y_test,
                color='#264653', alpha=0.65, edgecolors='white',
                linewidth=0.4, s=55, label='Actual data')
axes[1].plot(x_range, y_line, color='#E63946', linewidth=2.5,
             label='Linear Regression line', zorder=5)
axes[1].set_title('AFTER Fitting\nLinear Regression Line Through Data',
                  fontweight='bold')
axes[1].set_xlabel(f'{top_feat} (standardised)')
axes[1].set_ylabel('log(1 + Incidence per 1,000)')
axes[1].legend()

plt.suptitle('Visualization 6: Scatter Plot – Before & After Linear Regression Fit',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('viz6_scatter_before_after.png', dpi=150)
plt.show()

print()
print('📊 Interpretation:')
print('   BEFORE: raw scatter shows data points with no structure imposed.')
print('   AFTER : the red regression line captures the learned linear relationship')
print('   between the top predictor and malaria incidence, validating the model fit.')

---
## 7. Feature Importance (Random Forest)

In [ ]:
# ── 7. Feature Importance ─────────────────────────────────────────────────
importances = rf_model.feature_importances_
feat_imp    = pd.Series(importances, index=features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, max(4, len(features) * 0.45)))
bar_colors = ['#E63946' if v >= feat_imp.quantile(0.75) else '#2A9D8F'
              for v in feat_imp]
feat_imp.plot(kind='barh', ax=ax, color=bar_colors, edgecolor='white')
ax.set_title('Visualization 7: Feature Importance – Random Forest\n'
             '(Red = top 25% most important features)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('viz7_feature_importance.png', dpi=150)
plt.show()

print()
print('📊 Interpretation:')
print('   Red bars = the most powerful predictors of malaria incidence.')
print('   Health ministries should prioritise collecting these variables')
print('   in field surveys to enable accurate early-warning forecasting.')

---
## 8. Single-Row Prediction Script (Task 2 / API Ready)

In [ ]:
# ── 8. Reusable prediction function — plug directly into Task 2 API ────────
def predict_malaria_incidence(raw_feature_array):
    """
    Predict malaria incidence (per 1,000 population at risk).

    Args:
        raw_feature_array (array-like): 1-D array of feature values
            in the same order as feature_names.pkl
            (post log-transform of skewed columns, pre-standardisation)

    Returns:
        float: predicted malaria incidence on the original scale
    """
    model   = joblib.load('best_model.pkl')
    sc      = joblib.load('scaler.pkl')
    sample  = np.array(raw_feature_array).reshape(1, -1)
    scaled  = sc.transform(sample)
    log_pred = model.predict(scaled)[0]
    return float(np.expm1(log_pred))   # reverse log1p transform


# ── Demo: predict on the first row of the test set ─────────────────────────
sample_features = X_test[0]
prediction      = predict_malaria_incidence(sample_features)
actual          = float(np.expm1(y_test[0]))

print('━' * 62)
print('🔮  SINGLE-ROW PREDICTION DEMO')
print('━' * 62)
print(f'Input features        : {np.round(sample_features, 3)}')
print(f'\nPredicted incidence   : {prediction:>10.2f}  per 1,000 pop at risk')
print(f'Actual incidence      : {actual:>10.2f}  per 1,000 pop at risk')
print(f'Absolute error        : {abs(prediction - actual):>10.2f}')
print('━' * 62)
print('\n✅ predict_malaria_incidence() is production-ready.')
print('   Import this function into your Task 2 Flask/FastAPI endpoint.')

---
## Summary

| Model | MSE | MAE | R² |
|---|---|---|---|
| Linear Regression | *(run notebook to populate)* | | |
| Decision Tree | | | |
| Random Forest | | | |

**Key Findings:**
- Dataset loaded directly from Kaggle via `kagglehub`: `lydia70/malaria-in-africa`
- Real null values in prevention columns (bed nets, fever treatment, antenatal care) — handled by dropping rows with null target, then median imputation of features
- Categorical features (Country, Country Code) label-encoded; skewed numeric features log-transformed
- All features standardised with `StandardScaler` before training
- Three models trained and compared; best (lowest MSE) saved as `best_model.pkl`
- `predict_malaria_incidence()` ready for Task 2 API deployment

**Saved files:**
- `best_model.pkl` — best performing model (joblib)
- `scaler.pkl` — fitted StandardScaler
- `feature_names.pkl` — ordered feature list
- `viz1_missing_values.png` through `viz7_feature_importance.png`

**Dataset:** https://www.kaggle.com/datasets/lydia70/malaria-in-africa